# MIA — Kendi KKD Modelimizi Eğitme (v1 → v2)**Amaç:** Topluluk ağırlıklarından kurtulmak ve **gözlük + eldiven** gibi bugünkilitli olan sınıfları açmak.**Nasıl kullanılır:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU** seç.Sonra `Çalışma zamanı → Tümünü çalıştır`. Toplam ~2-4 saat (ücretsiz T4).**Çıktı:** `mia-ppe-vX.onnx` — indirip `apps/desktop/models/mia-ppe-yolov8s.onnx`üzerine yazarsın; uygulama başka değişiklik istemez.> DÜRÜSTLÜK KURALI: Aday model, eval kapısında mevcut modeli **geçmeden**> yayınlanmaz. Bu defter kapıyı otomatik uygular.

## 0) Ortam kontrolü — GPU var mı?

In [ ]:
!nvidia-smiimport torch; print("CUDA:", torch.cuda.is_available())# GPU YOKSA: Çalışma zamanı → Çalışma zamanı türünü değiştir → Donanım hızlandırıcı: T4 GPU

## 1) Kurulum

In [ ]:
%pip -q install ultralytics==8.3.* onnx onnxruntime roboflowimport ultralytics; ultralytics.checks()

## 2) Veri setiİki kaynak birleştirilir:**A. Temel (zorunlu):** Construction Site Safety — 10 sınıf (baret/yelek/maske + NO-*).Roboflow Universe'ten indirilir. Ücretsiz API anahtarı: roboflow.com → Settings → API Key.**B. Yeni sınıflar (gözlük + eldiven — kilidi açan kısım):** aşağıdaki 3. hücredeseçenekler var. En sağlıklısı: MIA saha verisi (masaüstü uygulama → Ayarlar →Saha Veri Toplama Modu) + CVAT'ta elle etiketleme.

In [ ]:
ROBOFLOW_API_KEY = ""  # ← roboflow.com → Settings → API Keyfrom roboflow import Roboflowrf = Roboflow(api_key=ROBOFLOW_API_KEY)# Temel veri seti (10 sınıf, ~2800 görüntü, CC BY 4.0)ds = (rf.workspace("roboflow-universe-projects")        .project("construction-site-safety")        .version(30)        .download("yolov8", location="/content/css"))print("Temel veri:", ds.location)

### 3) Gözlük + eldiven verisi (KİLİDİ AÇAN ADIM)Bu sınıflar temel veri setinde YOK. Üç yol:1. **MIA saha verisi (en değerli):** Uygulamada Ayarlar → Saha Veri Toplama Modu'nu   aç (KVKK onayı şart). Toplanan kareleri `mia-dataset` klasöründen Drive'a yükle,   CVAT/Label Studio'da `safety_glasses / gloves` etiketle. TR şantiye verisi asıl farkı yaratır.2. **Roboflow Universe'te hazır PPE veri setleri:** aşağıdaki hücre birkaç adayı indirir;   sınıf adları farklıysa `CLASS_MAP` ile bizim şemaya çevrilir.3. **İkisi birlikte** — önerilen.> Not: Eldiven/gözlük küçük nesnelerdir. Sınıf başına **en az 1500-2000** örnek> ve %30+ negatif (takılı DEĞİL) olmadan precision 0.80 hedefini tutturmak zordur.

In [ ]:
# İSTEĞE BAĞLI: ek PPE veri seti (gözlük/eldiven içerenler).# Roboflow Universe'te arama: "ppe gloves goggles detection"# Bulduğun projeyi buraya yaz; sınıf adlarını CLASS_MAP ile bizim şemaya çevir.EXTRA_DATASETS = [    # ("workspace-adi", "proje-adi", surum_no),]CLASS_MAP = {   # kaynak sınıf adı → MIA sınıf adı    "goggles": "Safety Glasses", "safety_goggles": "Safety Glasses", "glasses": "Safety Glasses",    "no_goggles": "NO-Safety Glasses", "no-glasses": "NO-Safety Glasses",    "gloves": "Gloves", "hand_gloves": "Gloves",    "no_gloves": "NO-Gloves", "bare_hands": "NO-Gloves",}for ws, proj, ver in EXTRA_DATASETS:    rf.workspace(ws).project(proj).version(ver).download("yolov8", location=f"/content/extra_{proj}")print("Ek veri setleri indirildi:", len(EXTRA_DATASETS))

### 4) MIA saha verini Drive'dan bağla (isteğe bağlı)`mia-dataset` klasörünü Google Drive'a yükledinse çalıştır.

In [ ]:
USE_DRIVE = False   # Drive'da MIA saha verin varsa True yapMIA_DATA_PATH = "/content/drive/MyDrive/mia-dataset"if USE_DRIVE:    from google.colab import drive; drive.mount("/content/drive")    import os; print("MIA saha verisi:", os.path.exists(MIA_DATA_PATH))

## 5) Veri hazırlığı — birleştir, böl, data.yaml üret

In [ ]:
import os, glob, random, shutil, yamlfrom pathlib import Path# MIA sınıf sırası — apps/desktop/renderer/js/ppe-registry.js ile SENKRON olmalı.# v1: mevcut 10 sınıf.  v2 için yorumu kaldır (yeni veri gerekir!).CLASSES = ["Hardhat","Mask","NO-Hardhat","NO-Mask","NO-Safety Vest",           "Person","Safety Cone","Safety Vest","machinery","vehicle"]# CLASSES += ["Safety Glasses","NO-Safety Glasses","Gloves","NO-Gloves"]OUT = Path("/content/mia_data")def collect(root):    pairs = []    for img in Path(root).rglob("*.jpg"):        if "images" not in img.parts: continue        lbl = Path(str(img).replace("/images/","/labels/")).with_suffix(".txt")        if lbl.exists(): pairs.append((img, lbl))    return pairspairs = collect("/content/css")print("Temel veri:", len(pairs))for p in glob.glob("/content/extra_*"):    ex = collect(p); pairs += ex; print(" +", p, len(ex))if USE_DRIVE and os.path.exists(MIA_DATA_PATH):    mia = collect(MIA_DATA_PATH); pairs += mia    print(" + MIA saha verisi:", len(mia), "(etiketleri CVAT'ta doğruladığından emin ol!)")def valid(lbl):    try: lines = Path(lbl).read_text().strip().splitlines()    except OSError: return False    if not lines: return False    for ln in lines:        p = ln.split()        if len(p) != 5: return False        try:            c = int(p[0]); vals = [float(x) for x in p[1:]]        except ValueError: return False        if not (0 <= c < len(CLASSES)) or any(v < 0 or v > 1.5 for v in vals): return False    return Truepairs = [(i,l) for i,l in pairs if valid(l)]print("Geçerli toplam:", len(pairs))assert len(pairs) > 500, "Veri çok az — indirme adımını kontrol et"random.Random(42).shuffle(pairs)n_val = int(len(pairs)*0.15)splits = {"val": pairs[:n_val], "train": pairs[n_val:]}for sp, items in splits.items():    (OUT/sp/"images").mkdir(parents=True, exist_ok=True)    (OUT/sp/"labels").mkdir(parents=True, exist_ok=True)    for i,(img,lbl) in enumerate(items):        stem = f"{sp}_{i:06d}"        shutil.copyfile(img, OUT/sp/"images"/(stem+".jpg"))        shutil.copyfile(lbl, OUT/sp/"labels"/(stem+".txt"))    print(sp, len(items))(OUT/"data.yaml").write_text(yaml.dump({    "path": str(OUT), "train": "train/images", "val": "val/images",    "nc": len(CLASSES), "names": CLASSES}, allow_unicode=True))print("✔ data.yaml hazır")

## 6) Eğitim~2-3 saat (T4, 100 epoch). Colab bağlantısı koparsa `resume=True` ile devam edebilirsin.Augmentasyon TR şantiye koşullarına göre: toz/ışık (hsv), açı, uzak kamera (scale).

In [ ]:
from ultralytics import YOLOmodel = YOLO("yolov8s.pt")   # COCO ön-eğitimli tabanresults = model.train(    data=str(OUT/"data.yaml"), epochs=100, imgsz=640, batch=16,    name="mia-ppe", project="/content/runs", patience=25,    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,    degrees=8, scale=0.5, fliplr=0.5,    mosaic=1.0, close_mosaic=15, seed=42,)

## 7) EVAL KAPISI — aday modeli mevcut modelle karşılaştır

In [ ]:
# Ultralytics'in kendi doğrulaması: sınıf bazlı precision/recall/mAPbest = "/content/runs/mia-ppe/weights/best.pt"m = YOLO(best)metrics = m.val(data=str(OUT/"data.yaml"), split="val")import numpy as npnames = m.namesp, r = metrics.box.p, metrics.box.rf1 = 2*p*r/np.maximum(1e-9, p+r)print(f"\n{'Sınıf':<20}{'P':>7}{'R':>7}{'F1':>7}")for i, n in names.items():    print(f"{n:<20}{p[i]:>7.2f}{r[i]:>7.2f}{f1[i]:>7.2f}")# KAPI KURALI: kritik ihlal sınıflarında precision >= 0.80CRITICAL = ["NO-Hardhat", "NO-Safety Vest"]gate_ok = Truefor n in CRITICAL:    idx = [i for i,v in names.items() if v == n]    if not idx:        print(f"⚠ {n} bu modelde yok"); continue    i = idx[0]    if p[i] < 0.80:        print(f"✘ KAPI: {n} precision {p[i]:.2f} < 0.80"); gate_ok = False    else:        print(f"✔ {n} precision {p[i]:.2f}")print("\n" + ("✔ EVAL KAPISI GEÇİLDİ — yayınlanabilir" if gate_ok      else "✘ KAPI GEÇİLEMEDİ — veri ekle / tekrar eğit. KISAYOL YOK."))

## 8) ONNX'e çevir ve indirKapı geçtiyse çalıştır. İnen dosyayı `apps/desktop/models/mia-ppe-yolov8s.onnx`üzerine kopyala → `npm start` ile duman testi → `npm run release:mac`.

In [ ]:
onnx_path = YOLO(best).export(format="onnx", imgsz=640, simplify=True)print("ONNX:", onnx_path)# Sınıf sırasını doğrula — uygulamadaki registry ile birebir olmalıimport onnxmm = onnx.load(onnx_path)meta = {p.key: p.value for p in mm.metadata_props}print("Model sınıfları:", meta.get("names"))print("BEKLENEN      :", {i: c for i, c in enumerate(CLASSES)})import hashlib, shutilshutil.copyfile(onnx_path, "/content/mia-ppe-v1.onnx")print("SHA256:", hashlib.sha256(open("/content/mia-ppe-v1.onnx","rb").read()).hexdigest())from google.colab import files; files.download("/content/mia-ppe-v1.onnx")

## 9) Yeni sınıfları uygulamada AÇMA (v2)Model gözlük/eldiven içeriyorsa (5. hücrede `CLASSES` genişletilmiş ve kapı geçilmişse):`apps/desktop/renderer/js/ppe-registry.js` içinde ilgili kaydı güncelle:```js{    key: "safety_glasses", status: "experimental",   // requires_training → experimental    okClass: "Safety Glasses", violationClass: "NO-Safety Glasses",    ...}```Aynısını `js/ppe-registry.js` (web) ve `workers/.../ppe_registry.py`'de yap — **üçü senkron**.Uygulamada kilit otomatik açılır; çip tıklanabilir olur. Saha doğrulaması bitince`experimental → supported` terfisi yapılır.